# Handwritten RNN — Character-Level Language Model from Scratch

This notebook trains a **Vanilla RNN from scratch** in PyTorch — without using `nn.RNN` or any higher-level recurrent API. Every step of the recurrence is written explicitly so you can see exactly what happens at each time step.

## What This Does
- Trains a **character-level language model** on a small text corpus
- Learns to predict the **next character** given all previous characters
- After training, **generates new text** by sampling from the model one character at a time

## Recurrence Formula
At every time step $t$, the model computes:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$
$$y_t = W_{hy} \cdot h_t + b_y$$

$h_t$ is the **hidden state** — the model's memory. $y_t$ holds raw **logit scores** over the vocabulary.

## Files
| File | Purpose |
|------|---------|
| `data_setup.py` | Builds the character vocabulary and encodes the corpus as integer indices |
| `handwritten_rnn.ipynb` | Model definition, training loop, and text generation (this notebook) |
| `handwritten_rnn.pt` | Saved model weights after training |

---
## Complete Reference Script
The cell below is the entire codebase as a **single runnable script**. The cells that follow break it down section by section with detailed explanations.

---
## Step-by-Step Breakdown

The cells below walk through the same code in structured, annotated sections.

---

### 1. Imports & Data Setup

This cell imports all required libraries and the processed data from `data_setup.py`.

| Import | Source | Purpose |
|--------|--------|---------|
| `torch` | PyTorch | Tensors, autograd, model saving |
| `torch.nn` | PyTorch | Neural network layers (`nn.Linear`, `nn.Embedding`, etc.) |
| `torch.nn.functional` | PyTorch | Stateless functions: `softmax`, `cross_entropy` |
| `text` | `data_setup` | The raw string corpus |
| `chars` | `data_setup` | Sorted list of unique characters in the corpus |
| `vocab_size` | `data_setup` | Total number of unique characters (length of `chars`) |
| `char_to_idx` | `data_setup` | Dict mapping each character → integer index |
| `idx_to_char` | `data_setup` | Dict mapping each integer index → character |
| `encode` | `data_setup` | Function: string → `torch.LongTensor` of indices |
| `decode` | `data_setup` | Function: list of indices → string |
| `data` | `data_setup` | The full corpus pre-encoded as a `LongTensor` |

`data_setup.py` uses this corpus:
```
the quick brown fox jumps over the lazy dog.
she sells seashells by the seashore.
how much wood would a woodchuck chuck if a woodchuck could chuck wood.
peter piper picked a peck of pickled peppers.
the rain in spain stays mainly on the plain.
```
It strips the text, lowercases it, computes the character vocabulary, and encodes the entire string as integer indices stored in `data`.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from data_setup import text, chars, vocab_size, char_to_idx, idx_to_char, encode, decode, data

Corpus length: 8077 characters
Vocabulary size: 50 unique characters
Vocabulary: ['\n', ' ', "'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '‘', '’', '“', '”']

Encoded data shape: torch.Size([8077])
First 20 chars encoded: [37, 24, 19, 1, 39, 27, 24, 1, 32, 20, 39, 39, 24, 37, 1, 34, 25, 1, 20, 23]
Decoded back: 're: the matter of ad'


### 2. Reproducibility — Random Seed

`torch.manual_seed(42)` fixes PyTorch's internal random number generator (RNG).

**Why it matters:**  
- **Weight initialisation** in `nn.Linear` and `nn.Embedding` uses the RNG — the seed guarantees the same starting weights every run.  
- **Batch sampling** in `get_batch` uses `torch.randint` — the seed makes training sequences deterministic.  

Set this once, at the start of the notebook, before any model or data operations.

In [2]:
torch.manual_seed(42)

### 3. Model Architecture — `VanillaRNN`

This class implements a Vanilla RNN **by hand**: no `nn.RNN`, no `nn.GRU`. The recurrence loop runs explicitly in Python.

#### Layers

| Attribute | Type | Shape | Purpose |
|-----------|------|-------|---------|
| `embed` | `nn.Embedding` | `vocab_size → hidden_size` | Converts a character index into a dense floating-point vector |
| `Wxh` | `nn.Linear` | `hidden_size → hidden_size` | Weights for the **current input** $x_t$ (includes bias $b_h$) |
| `Whh` | `nn.Linear` (no bias) | `hidden_size → hidden_size` | Weights for the **previous hidden state** $h_{t-1}$ |
| `Why` | `nn.Linear` | `hidden_size → vocab_size` | Projects the hidden state to vocabulary logit scores |

#### Recurrence — Line by Line
```python
h = torch.tanh(self.Wxh(x_t) + self.Whh(h))
```
This is the explicit recurrence step:
- `self.Wxh(x_t)` → $W_{xh} \cdot x_t + b_h$ — how much the *current character* influences the new state
- `self.Whh(h)` → $W_{hh} \cdot h_{t-1}$ — how much *memory from the past* carries forward
- `tanh(...)` squashes the sum into $(-1, +1)$, acting as the non-linearity

The hidden state $h$ is **the model's entire memory** — it is the only thing that carries information between time steps.

#### `forward(x, h=None)`
- `x` shape: `(batch, seq_len)` — batch of integer character sequences
- If `h` is `None` (start of a sequence), it initialises to a zero tensor
- Loops over each time step `t`, runs the recurrence, collects output logits
- Returns:
  - `logits`: shape `(batch, seq_len, vocab_size)` — one distribution per step
  - `h`: the final hidden state (can be passed back in to continue generation)

#### Scratch Cell — Inspecting `nn.Embedding`

The cell below is a quick, standalone experiment (not part of the main pipeline). It creates a throwaway embedding table with `vocab_size=10` and `hidden_size=10` and prints its raw weight matrix.

```python
hh = nn.Embedding(10, 10)
print(hh.weight)
```

- `hh.weight` is a learnable `(10, 10)` parameter tensor, randomly initialised
- Each **row** is the dense vector representation for one index (e.g. row 3 is the embedding for index 3)
- This is exactly what `self.embed` does inside `VanillaRNN` — this cell just isolates it so you can see the raw values before any training happens

In [22]:
hh = nn.Embedding(10, 10)
print(hh.weight)

Parameter containing:
tensor([[-9.1632e-01,  1.5206e-01,  1.8018e+00,  2.7550e-01, -1.7786e-01,
          1.0270e+00,  7.1677e-01,  1.1661e+00, -1.2166e+00, -3.0719e-01],
        [-4.6438e-01,  4.9539e-01,  1.4034e-01,  1.3721e+00, -1.0607e+00,
          1.3158e+00, -1.3119e+00,  4.2816e-01,  7.7387e-01, -7.7646e-01],
        [ 5.5302e-01,  1.7433e-01,  1.1156e+00, -6.7310e-01, -1.2237e+00,
         -1.4542e-01,  2.2951e+00,  7.4706e-01,  7.3261e-03,  8.2912e-02],
        [ 8.0861e-01, -7.5617e-01, -6.9472e-02, -1.0289e+00, -4.3209e-01,
          6.1592e-01,  2.3803e+00, -3.8865e-01,  6.3359e-02,  6.6996e-01],
        [ 2.3600e-01, -8.9266e-01,  2.8420e-01, -4.1180e-01, -6.6696e-01,
         -5.7397e-01,  4.7614e-01,  1.3837e+00,  3.2735e-01,  5.8986e-01],
        [ 7.0775e-01,  2.6208e-01, -1.0608e+00,  1.1062e+00,  7.0039e-01,
         -1.7157e+00,  1.2990e+00, -6.9876e-01, -9.2180e-02, -1.3482e+00],
        [-6.2308e-01, -1.2786e+00,  1.3327e+00, -2.3249e+00, -8.7237e-01,
          

In [3]:
class VanillaRNN(nn.Module):
    """
    Hand-written recurrence — no nn.RNN black box.
    At each step: h_t = tanh(Wxh @ x_t + Whh @ h_{t-1} + bh)
                  y_t = Why @ h_t + by
    """
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Embedding: turn char index into a dense vector
        self.embed = nn.Embedding(vocab_size, hidden_size)
        # The actual recurrence weights (this is the "by hand" part)
        self.Wxh = nn.Linear(hidden_size, hidden_size)
        self.Whh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Why = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        # x: (batch, seq_len) of char indices
        batch, seq_len = x.shape
        if h is None:
            h = torch.zeros(batch, self.hidden_size, device=x.device)

        embedded = self.embed(x)  # (batch, seq_len, hidden_size)
        logits = []
        for t in range(seq_len):
            x_t = embedded[:, t, :]
            h = torch.tanh(self.Wxh(x_t) + self.Whh(h))  # <-- the recurrence, explicit
            y_t = self.Why(h)
            logits.append(y_t.unsqueeze(1))
        logits = torch.cat(logits, dim=1)  # (batch, seq_len, vocab_size)
        return logits, h

### 4. Helper Functions

#### `get_batch(data, seq_len, batch_size)`

Produces a random mini-batch from the encoded corpus.

```
data:  [ 20, 7, 4, 12, 28, 3, ... ]   ← full encoded corpus (LongTensor)
         ↑ pick random start indices
x:     [ data[s : s+seq_len] ]         ← input sequences
y:     [ data[s+1 : s+seq_len+1] ]     ← targets = x shifted right by 1
```

The shift-by-one is the **language modelling objective**: given characters at positions $1 \ldots n$, predict characters at positions $2 \ldots n+1$. Every position produces a supervised training signal.

Both `x` and `y` have shape `(batch_size, seq_len)`.

---

#### `generate(model, start_char, length=100)`

Generates new text **autoregressively** — each predicted character becomes the next input.

**Step-by-step:**
1. Convert `start_char` to its integer index and wrap in a `(1, 1)` tensor
2. Run the model to get `logits` over the vocabulary
3. Apply `F.softmax` to turn logits into a probability distribution
4. **Sample** a character with `torch.multinomial` — this is stochastic, not greedy (argmax), so output varies between calls
5. Append the sampled character to `result`, feed it back as `x`
6. Pass the current hidden state `h` forward so the model retains context
7. Repeat `length` times

The model is set to `eval()` mode during generation (disables dropout if any) and back to `train()` after.

In [4]:
def get_batch(data, seq_len, batch_size):
    """Sample random chunks of text for training."""
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[s:s+seq_len] for s in starts])
    y = torch.stack([data[s+1:s+seq_len+1] for s in starts])  # shifted by 1 = "predict next char"
    return x, y

def generate(model, start_char, length=100):
    model.eval()
    idx = char_to_idx[start_char]
    x = torch.tensor([[idx]])
    h = None
    result = [start_char]
    with torch.no_grad():
        for _ in range(length):
            logits, h = model(x, h)
            probs = F.softmax(logits[0, -1], dim=0)
            idx = torch.multinomial(probs, 1).item()
            result.append(idx_to_char[idx])
            x = torch.tensor([[idx]])
    model.train()
    return ''.join(result)


#### Scratch Cells — Inspecting `get_batch` Output

The two cells below are quick, standalone checks (not part of the main pipeline) used to verify `get_batch` produces tensors of the expected shape before wiring it into the training loop.

```python
x, y = get_batch(data, 25, 16)
```
Calls `get_batch` directly with `seq_len=25` and `batch_size=16` — the same values used later in the real training loop — and inspects the result on its own.

```python
y.shape
```
Confirms the target tensor `y` has shape `(16, 25)` — 16 sequences of 25 characters each — matching `x`'s shape as expected for the shift-by-one language modelling setup described above.

In [29]:
x, y = get_batch(data, 25, 16)

In [33]:
y.shape

torch.Size([16, 25])

### 5. Training Loop

#### Hyperparameters

| Parameter | Value | Notes |
|-----------|-------|-------|
| `hidden_size` | 64 | Dimensionality of the hidden state vector |
| `seq_len` | 25 | Characters per training sequence |
| `batch_size` | 16 | Sequences per gradient update |
| `lr` | 0.01 | Adam learning rate |
| `epochs` | 2000 | Total number of gradient steps |

#### Per-Epoch Training Steps

1. **Sample a batch** — `get_batch` draws `batch_size` random sequences from `data`
2. **Forward pass** — `model(x)` runs the full recurrence and returns `logits` of shape `(batch, seq_len, vocab_size)`
3. **Compute loss** — `F.cross_entropy` flattens the batch and sequence dimensions to `(batch × seq_len, vocab_size)` and computes average negative log-likelihood against the targets `y`
4. **Zero gradients** — `optimizer.zero_grad()` clears accumulated gradients from the previous step
5. **Backward pass** — `loss.backward()` runs Backpropagation Through Time (BPTT), computing gradients for all weights
6. **Gradient clipping** — `clip_grad_norm_(model.parameters(), 5.0)` rescales gradients if their global norm exceeds 5.0
7. **Optimizer step** — Adam updates all parameters

#### Why Gradient Clipping?

Vanilla RNNs suffer from **exploding gradients** during BPTT. Because the same weight matrix $W_{hh}$ is multiplied through every time step:

$$\frac{\partial L}{\partial W_{hh}} \propto (W_{hh})^T$$

If any singular value of $W_{hh}$ is $> 1$, gradients grow **exponentially** with sequence length and cause numerical instability or divergence. Clipping their global norm to 5.0 keeps training stable without changing gradient direction.

#### Loss Function — Cross-Entropy

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid x_{\leq i})$$

At every position in every sequence, the model is penalised for assigning low probability to the correct next character. A perfect model knowing only the training text would reach a loss near 0; an untrained model on a 30-character vocabulary starts near $\ln(30) \approx 3.4$.

#### Saving the Model

After training, `torch.save(model.state_dict(), "handwritten_rnn.pt")` saves only the **learned weights** (not the model class itself). To reload:
```python
model = VanillaRNN(vocab_size, hidden_size)
model.load_state_dict(torch.load("handwritten_rnn.pt"))
```

In [8]:
hidden_size = 128
seq_len = 100
batch_size = 16
lr = 0.01
epochs = 3000

model = VanillaRNN(vocab_size, hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

print("=== Before training ===")
print(generate(model, 't', 500))
print()

losses = []
for epoch in range(epochs):
    x, y = get_batch(data, seq_len, batch_size)
    logits, _ = model(x)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    # Gradient clipping — vanilla RNNs explode without it
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()

    losses.append(loss.item())
    if epoch % 400 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:4d} | loss {loss.item():.4f} | sample: {generate(model, 't', 60)!r}")

print("\n=== After training ===")
print(generate(model, 't', 500))

torch.save(model.state_dict(), "handwritten_rnn.pt")
print(f"\nFinal loss: {losses[-1]:.4f}")

=== Before training ===
t9 ,x“-4
y8/g3.9
eg“”,'’sr.di /pz‘-’lw.spdrnp8:5mqo“gk’.”“’ugeuz0wrq56(nii0.z“yzlc.,6/’n(9v1q/s6
g0z8'uswk lojo.hd,/a8a)7lz54pr4pm)ic0’./0-m4'r”x2a4ff‘cwkg“no2:“b3ml3yy fc”2v7’i1’7y4k)m42sn(93'x‘0l4l9kw0:4d’)nzufd2’i'lq'fm-3e’y“fsuo(nd0‘‘qw9“ 3q:05xe06”’’me1wakr6e“‘v0w,747c5g4qce9t(t'q8ros9”t.z(izv0e674,9sc8'cv,us('emq8'4x5b-b8)s‘ k)-4('
n'9dma'
d48irws
h””u'.9'oji41rupd12a”975v64 :0(rs9le
6muvb5'e:383plfnu6,l vv,’w”'y6-“,l/vpihqv-)no ‘.z2”m”0701o-h).sl3-“”7ca-a:s-vlq”7w“g6mqvk0fn:xf8siwk7 tc1“0o

Epoch    0 | loss 3.9330 | sample: "t7fg0e1qojl’hx\n426f)-9er:am x96\n9”\np'c.”\n\n3qi,\n'amalind2wjq90"
Epoch  400 | loss 0.5398 | sample: 't to never of the existence of the foreding to cinistrats fol'
Epoch  800 | loss 0.8307 | sample: 't he tent unculabrations, to obtaining by fctmovent agency at'
Epoch 1200 | loss 0.7032 | sample: 'tanied the statushif a letter ableany of a coner, was and ade'
Epoch 1600 | loss 0.7557 | sample: 'the eign affairs of the illegal app